# Suite TST — The TypeScript Types dialect

TypeScript's types, typescript-estree's type nodes, the sublanguage that `as`, `satisfies`, `<T>x`, an annotation, a
type argument and a type parameter's constraint and default hold: the dialect `Expressions.TypeScript.Types` of the
TypeScript family, whose `Term` was TypeScript5's `Type` union, its branches unchanged, its kinds mbse-programs' 37 type
nodes, its holder, its binders refining `Let`, its editions, its printing, and its evaluation, which refuses each kind by
name (mbse-basis [CAP-277: TypeScript's types and destructuring patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-277.md);
[CAP-228: the TypeScript5 expression dialect](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-228.md)).

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as TypeScriptPatterns from "@mbse/expressions/TypeScript/Patterns";
import * as TypeScriptTypes from "@mbse/expressions/TypeScript/Types";
import { Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const { NotImplementedError, ValueError } = Errors;
const [TS, TV, TSText] = [TypeScript5.Expressions, TypeScript5.Simplifiers, TypeScript5.Text];
const [TST, TSTText, TSP] = [TypeScriptTypes.Expressions, TypeScriptTypes.Text, TypeScriptPatterns.Expressions];

const store: any = new Builders.Store(); // the framework, the standard dialects TypeScript5's domains use, and its family
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, Unicode.install, TypeScript5.install,
  TypeScriptTypes.install, TypeScriptPatterns.install]) {
  store.transaction(install);
}
store.transaction(TypeScriptTypes.define); // the versions of TypeScript 5 the Types dialect covers
const Term: any = store.registered("Expressions.Framework.Term");
const READER = new Map([TS.DIALECT, TST.DIALECT, TSP.DIALECT].map((d) => [d.union, d])); // the dialect whose union holds a term

/** The holder of the term `spec` writes, `dialect`'s own (the Types dialect's unless given), written in a transaction. */
function held(spec: unknown, dialect: F.Dialect = TST.DIALECT): any {
  return store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec))));
}

const typed = (spec: unknown): any => F.term(held(spec)); // a term of the Types dialect, held by its holder
const ts = (spec: unknown): any => F.term(held(spec, TS.DIALECT)); // a term of TypeScript5, held by its holder
const names = (found: Iterable<string>): string[] => [...found].sort();

/** A spec of a reference to the type `name`, in any union that has type references. */
function reference(name: string): (t: any) => any {
  return (t: any) => t.tstypereference((r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name(name))));
}

const [T_, U_, K_] = [reference("T"), reference("U"), reference("K")];
const number = (t: any) => t.tsnumberkeyword((k: any) => k);
const x = (e: any) => e.identifier((i: any) => i.name("x")); // `x`, in any union that has identifiers

<a id="TST-01"></a>
## TST-01 · The Types dialect's union, its kinds' schemas in mbse-programs' shapes, its helpers, the properties that hold it as a subterm, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
// Its Term: mbse-programs' 37 type nodes, its own, then the six terms of TypeScript5 a type holds, and invalid syntax, as
// TypeScript5's `Type` union had them; each own kind's schema is named by its path, in mbse-programs' case.
const union: string[] = store.registered(TST.DIALECT.union).branches.map((b: any) => b.name);
assert(same(union, [...TST.TAGS, ...TST.TERMS, "error", "missing"]) && TST.TAGS.length === 37);
assert(same(TST.TERMS, ["identifier", "thisexpression", "literal", "unaryexpression", "templateliteral", "objectexpression"]));
assert(TST.DIALECT.name() === "TypeScript Types" && TST.DIALECT.tag === "typescript_types");
assert(TST.DIALECT.path === "TypeScript.Types" && TST.DIALECT.union === "Expressions.TypeScript.Types.Term");
const kinds = new Map<string, any>(store.registered(TST.DIALECT.union).branches.map((b: any) => [b.name, b.schema]));
assert(same(TST.TAGS.slice(0, 3).map((tag) => kinds.get(tag).name), [
  "Expressions.TypeScript.Types.TSAnyKeyword", "Expressions.TypeScript.Types.TSUnknownKeyword",
  "Expressions.TypeScript.Types.TSNumberKeyword"]));
assert(TST.TAGS.every((tag) => kinds.get(tag).name.startsWith("Expressions.TypeScript.Types.TS")));
assert(TST.TERMS.every((tag) => kinds.get(tag).name.startsWith("Expressions.TypeScript.TypeScript5."))); // TypeScript5's
assert(kinds.get("identifier") === store.registered(TS.DIALECT.union).branches.find((b: any) => b.name === "identifier").schema);
assert(equal(new Map(["tsqualifiedname", "tstypereference", "tsmappedtype", "tsfunctiontype", "tsconditionaltype", "tsimporttype",
  "tstypepredicate"].map((tag) => [tag, [...kinds.get(tag).properties.keys()]])), map({
  tsqualifiedname: ["left", "right"], tstypereference: ["typeName", "typeArguments"],
  tsmappedtype: ["name", "value", "body", "readonly", "key", "nameType", "optional"], // Let's first
  tsfunctiontype: ["name", "value", "body", "typeParameters", "params", "returnType"],
  tsconditionaltype: ["checkType", "extendsType", "trueType", "falseType"],
  tsimporttype: ["source", "options", "qualifier", "typeArguments"],
  tstypepredicate: ["asserts", "parameterName", "typeAnnotation"] })));
// Its helpers, only its kinds hold them: an object type's members, the union of its signatures and comments; TypeScript5
// keeps the helpers expressions hold too (an annotation, a type parameter, its declaration and instantiation).
assert(same(TST.HELPERS, ["TSPropertySignature", "TSMethodSignature", "TSCallSignatureDeclaration",
  "TSConstructSignatureDeclaration", "TSIndexSignature", "Comment"]));
const TypeElement: any = store.registered("Expressions.TypeScript.Types.TypeElement");
assert(same(TypeElement.branches.map((b: any) => b.name), ["tspropertysignature", "tsmethodsignature", "tscallsignaturedeclaration",
  "tsconstructsignaturedeclaration", "tsindexsignature", "comment"]));
assert(same(TypeElement.branches.map((b: any) => b.schema.name), TST.HELPERS.map((h) => `Expressions.TypeScript.Types.${h}`)));
assert(kinds.get("tstypeliteral").properties.get("members").schema.element === TypeElement);
for (const name of ["TSTypeAnnotation", "TSTypeParameter", "TSTypeParameterDeclaration", "TSTypeParameterInstantiation"]) {
  assert(store.registered(`Expressions.TypeScript.TypeScript5.${name}`) !== null);
}
// The framework reads a type as a term: a kind's subterms and containers holding its Term are its arguments; its helpers
// hold them too, where TypeScript5's annotation and instantiation hold a type.
const Types: any = store.registered(TST.DIALECT.union);
assert(equal(F._parts(kinds.get("tsuniontype")), [[], ["types"], []]) && equal(F._parts(kinds.get("tsarraytype")), [["elementType"], [], []]));
assert(equal(F._parts(kinds.get("tsmappedtype")), [["value", "body", "nameType"], [], ["name", "readonly", "key", "optional"]]));
assert(equal(F._parts(kinds.get("tsfunctiontype")), [["value", "body"], ["params"], ["name", "typeParameters", "returnType"]]));
assert(kinds.get("tsfunctiontype").properties.get("params").schema.element === store.registered(TSP.DIALECT.union)); // Patterns'
const annotation: any = store.registered("Expressions.TypeScript.TypeScript5.TSTypeAnnotation");
assert(annotation.properties.get("typeAnnotation").schema === Types && equal(F._parts(annotation), [["typeAnnotation"], [], []]));
assert(store.registered("Expressions.TypeScript.TypeScript5.TSTypeParameterInstantiation").properties.get("params").schema.element === Types);
// Where TypeScript5 holds a type: `as`, `satisfies` and `<T>x` hold its Term; an annotation and a type argument too
const expressions = new Map<string, any>(store.registered(TS.DIALECT.union).branches.map((b: any) => [b.name, b.schema]));
assert(["tsasexpression", "tssatisfiesexpression", "tstypeassertion"].every(
  (tag) => expressions.get(tag).properties.get("typeAnnotation").schema === Types));
// Its holder, refining the framework's Expression, its root narrowed to its Term: validated, linked by HasRoot,
// round-tripped
const Holder: any = store.registered("Expressions.TypeScript.Types.Expression");
assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
assert(Holder.properties.get("root").schema === Types && same([...Holder.adjacencies.keys()], ["abi"]) && Holder.validate().length === 0);
const mapped = held((t: any) => t.tsmappedtype((m: any) => m.key((i: any) => i.name("K")).value(T_).body(K_)));
assert(Validators.Validate(store)(Holder, mapped).length === 0 && TST.DIALECT.validate(mapped).length === 0);
const linked = store.transaction(() => store.Schemas.OfExpression((s: any) => s.name("tst.mapped").expression(
  (a: any) => a.insert((e: any) => e.expression(mapped)))));
assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, mapped)), [true]));
for (const [encode, decode] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as any[]) {
  assert(F.same(decode(Holder, encode(Holder, mapped)).root, mapped.root));
  assert(F.same(decode(Types, encode(Types, F.term(mapped))), F.term(mapped)));
}
// Its install runs before or after its family's: each names the others' unions and helpers, created where not installed
const alone: any = new Builders.Store();
for (const install of [Framework.install, TypeScriptPatterns.install, TypeScriptTypes.install, TypeScript5.install]) {
  alone.transaction(install);
}
assert(same(alone.registered(TST.DIALECT.union).branches.map((b: any) => b.name), union));
assert(same([...alone.registered("Expressions.TypeScript.TypeScript5.TSTypeAnnotation").properties.keys()], ["typeAnnotation"]));

<a id="TST-02"></a>
## TST-02 · Each kind's checks, and an identifier in a type, a reference never reported unbound; forms and `make`; inference and evaluation, which refuse or know nothing of a type

In [ ]:
// Each kind's own checks, as mbse-programs has them, within a type held by its holder or by an expression
for (const [spec, problems] of [
  [(t: any) => t.tsuniontype((u: any) => u.types([T_])), ["a tsuniontype needs two types"]],
  [(t: any) => t.tsintersectiontype((u: any) => u.types([T_, U_])), []],
  [(t: any) => t.tstemplateliteraltype((u: any) => u.quasis([(q: any) => q.raw("a")]).types([T_])),
    ["a tstemplateliteraltype needs one more quasi than types"]],
  [(t: any) => t.tsarraytype((a: any) => a), ["a tsarraytype needs an elementType"]], // a subterm it needs
  [(t: any) => t.tsmappedtype((m: any) => m.key((i: any) => i.name("K"))), ["a tsmappedtype needs a value"]],
  [(t: any) => t.tsmappedtype((m: any) => m.key((i: any) => i.name("K K")).value(T_)), ["an identifier's name cannot be 'K K'"]],
  [(t: any) => t.tstypeoperator((o: any) => o.operator("keyof")), []], // its type, optional
  [(t: any) => t.tsimporttype((i: any) => i.source((s: any) => s.literal((li: any) => li.value((v: any) => v.as_scalar("m"))))), []],
  [(t: any) => t.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([]))), ["a tstypeparameterdeclaration needs a type parameter"]],
  [(t: any) => t.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))]))), []],
  [(t: any) => t.tsfunctiontype((f: any) => f.params([(p: any) => p.identifier((i: any) => i.name("a b"))])),
    ["argument 0: an identifier's name cannot be 'a b'"]], // a parameter, a Patterns term
  [(t: any) => t.tstypeliteral((q: any) => q.members([(m: any) => m.tspropertysignature((s: any) => s.typeAnnotation(
    (a: any) => a.typeAnnotation(T_)))])), ["a tstypeliteral needs a members 0 key"]], // a helper's subterm, by its path
  [(t: any) => t.tsfunctiontype((f: any) => f.returnType((a: any) => a)), ["a tsfunctiontype needs a returnType typeAnnotation"]],
  [(t: any) => t.tsliteraltype((q: any) => q.literal((li: any) => li.literal((v: any) => v.value((n: any) => n.as_scalar("s")).quote("`")))),
    ["literal: a literal's quote cannot be '`'"]], // a term of TypeScript5's, as it reads it
  [(t: any) => t.tsconditionaltype((c: any) => c.checkType(T_).extendsType(U_).trueType(T_)), ["a tsconditionaltype needs a falseType"]],
  [(t: any) => t.missing((m: any) => m.kind("type")), ["missing type"]],
  [(t: any) => t.tsfunctiontype((f: any) => f.params([(p: any) => p.identifier((i: any) => i.name("a").typeAnnotation(
    (a: any) => a.typeAnnotation((e: any) => e.error((r: any) => r.text("1 1")))))])),
    ["argument 0: invalid syntax '1 1'"]], // a type within a parameter's annotation, which its identifier checks
  [(t: any) => t.tsfunctiontype((f: any) => f.params([(p: any) => p.identifier((i: any) => i.name("a").typeAnnotation((a: any) => a))])), []],
] as [unknown, string[]][]) {
  assert(same(TST.DIALECT.validate(typed(spec)), problems), globalThis.JSON.stringify([TST.DIALECT.validate(typed(spec)), problems]));
}
// An identifier in a type names a type, or a value whose type `typeof` reads, declared by what a term does not model: a
// reference, never reported unbound, nor free; its name is checked as an identifier's
assert(TST.DIALECT.kinds().get("identifier")!.role === F.REFERENCE && !TST.DIALECT.kinds().get("identifier")!.lexical);
assert(TST.DIALECT.kindNamed("identifier") !== TS.DIALECT.kinds().get("identifier") && TS.DIALECT.kinds().get("identifier")!.lexical);
const query = typed((t: any) => t.tstypequery((q: any) => q.exprName((e: any) => e.identifier((i: any) => i.name("value")))));
assert(TST.DIALECT.validate(query).length === 0 && SY.free(query, TST.DIALECT).size === 0 && TS.DIALECT.validate(query).length === 0);
assert(same(TST.DIALECT.validate(typed((t: any) => t.tstypereference((r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name("1")))))), [
  "typeName: an identifier's name cannot be '1'"]));
const cast = ts((u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation(T_))); // `x as T`: x a value's name, T a type's
assert(same(TS.DIALECT.validate(cast), ["expression: identifier 'x' is not bound"]) && same(names(SY.free(cast, TS.DIALECT)), ["x"]));
// Its kinds read and written by their forms: a type's arguments are its subterms, its containers' terms, its helpers'
const every = typed((t: any) => t.tsuniontype((u: any) => u.types([...TST.KEYWORDS.map((keyword) => (k: any) => k[keyword]((h: any) => h)), T_,
  (k: any) => k.tstupletype((p: any) => p.elementTypes([(e: any) => e.tsnamedtuplemember((m: any) => m.label((i: any) => i.name("a")).elementType(T_))])),
  (k: any) => k.tsconditionaltype((c: any) => c.checkType(T_).extendsType((e: any) => e.tsinfertype((i: any) => i.typeParameter(
    (p: any) => p.name((n: any) => n.name("V"))))).trueType(T_).falseType(U_))])));
for (const node of F.walk(every)) {
  const reader = READER.get(F.dialectOf(node))!; // made by the dialect whose union holds it, in its holder
  assert(F.same(F.term(held(reader.make(F.form(node)), reader)), node));
}
assert(F.form(typed((t: any) => t.tsarraytype((a: any) => a.elementType(T_)))).arguments[0] !== null);
const form = F.form(typed(T_));
assert(form.kind === "tstypereference" && form.attributes.size === 0 && form.arguments.length === 1); // its name, a term
assert(F.form(form.arguments[0]).kind === "identifier" && equal(F.form(form.arguments[0]).attributes, map({ name: "T" })));
const generic = F.form(typed((t: any) => t.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])))));
const parameters = generic.attributes.get("typeParameters") as F.Part; // Let's empty, then a type parameter's
const parameter = (parameters.attributes.get("params") as F.Part[])[0]!;
assert(generic.kind === "tsfunctiontype" && same([...generic.attributes.keys()], ["typeParameters"]) && same(generic.arguments, [null, null, null, null]));
assert(parameters.kind === "tstypeparameterdeclaration" && parameters.size === 2 && parameter.kind === "tstypeparameter" && parameter.size === 2);
assert(equal((parameter.attributes.get("name") as F.Part).attributes, map({ name: "T" })) && (parameter.attributes.get("name") as F.Part).size === 0);
raises(ValueError, () => TST.DIALECT.make(new F.Form("call")), "TypeScript Types has no kind 'call'");
// Inference knows nothing of a type; evaluation refuses each kind by its dialect's name, as TypeScript5's evaluator does
assert(TST.DIALECT.infer(typed(T_)).name() === "any");
for (const [spec, message] of [[T_, "TypeScript Types tstypereference is not simplified"],
  [(t: any) => t.tsanykeyword((h: any) => h), "TypeScript Types tsanykeyword is not simplified"],
  [(t: any) => t.identifier((i: any) => i.name("T")), "TypeScript Types identifier is not simplified"]] as [unknown, string][]) {
  raises(NotEvaluated, () => TV.OfAny(typed(spec)).evaluate(), message);
}
raises(NotEvaluated, () => TV.OfAny(typed((t: any) => t.missing((m: any) => m))).evaluate(), "TypeScript Types missing is not simplified");
assert(TV.OfAny(ts((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(1.0))))).evaluate() === 1.0); // TypeScript5's, as before

<a id="TST-03"></a>
## TST-03 · Its binders refine `Let`: a mapped type's constraint and type, Let's `value` and `body`, its key bound in its name type and its type; a function type's and a constructor type's type parameters and parameters within them; a conditional type's `infer` declarations in its true type, by its scopes; a signature's parameters within it

In [ ]:
const LET: any = store.registered("Expressions.Framework.Let");
assert(same(TST.TAGS.filter((tag) => F.isBinder(kinds.get(tag))), ["tsfunctiontype", "tsconstructortype", "tsmappedtype"]));
assert(["tsmappedtype", "tsfunctiontype", "tsconstructortype"].every((tag) => kinds.get(tag).refines === LET));
assert(kinds.get("tsmappedtype").properties.get("value").schema === Types && kinds.get("tsmappedtype").properties.get("body").schema === Types); // its own
assert(kinds.get("tsfunctiontype").properties.get("value").schema === Term); // Let's, unset
assert(!F.isBinder(kinds.get("tsconditionaltype")) && !F.isBinder(TypeElement.branches[1].schema));
assert(TST.TAGS.every((tag) => TST.DIALECT.kinds().get(tag)!.scopes !== null)); // each binds by its scopes, none as a let
const scopesOf = (tag: string, term: any): string[][] => TST.DIALECT.kinds().get(tag)!.scopes!(F.kindOf(term)[1]).map(names);
// What each argument has bound: a mapped type's key in its name type and its type, not in its constraint
const keyed = typed((t: any) => t.tsmappedtype((m: any) => m.key((i: any) => i.name("K")).value(K_).body(K_).nameType(K_)));
assert(equal(scopesOf("tsmappedtype", keyed), [[], ["K"], ["K"]]));
assert(same([...F.form(keyed).attributes.keys()], ["key"]) && F.form(keyed).arguments.length === 3);
assert(TST.DIALECT.validate(keyed).length === 0 && SY.free(keyed, TST.DIALECT).size === 0);
const fn = typed((t: any) => t.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).params([
  (p: any) => p.identifier((i: any) => i.name("a").typeAnnotation((b: any) => b.typeAnnotation(T_))),
  (p: any) => p.restelement((r: any) => r.argument((i: any) => i.identifier((j: any) => j.name("rest"))))]).returnType(
  (r: any) => r.typeAnnotation((p: any) => p.tstypepredicate((q: any) => q.parameterName((n: any) => n.identifier((i: any) => i.name("a"))))))));
const scopes = scopesOf("tsfunctiontype", fn);
assert(scopes.length === F.form(fn).arguments.length && scopes.length === 7 && scopes.every((s) => same(s, ["T", "a", "rest"])));
assert(TST.DIALECT.validate(fn).length === 0 && SY.free(fn, TST.DIALECT).size === 0); // its parameters bound within it
const constructor = typed((t: any) => t.tsconstructortype((f: any) => f.params([x]).returnType((r: any) => r.typeAnnotation(T_))));
assert(equal(scopesOf("tsconstructortype", constructor), [["x"], ["x"], ["x"], ["x"]]));
// A conditional type binds its `infer` declarations in its true type, but those of a conditional type within its extends
const conditional = typed((t: any) => t.tsconditionaltype((c: any) => c.checkType(T_).extendsType((e: any) => e.tsarraytype((a: any) => a.elementType(
  (i: any) => i.tsinfertype((f: any) => f.typeParameter((p: any) => p.name((n: any) => n.name("V")))))))
  .trueType(reference("V")).falseType((e: any) => e.tsconditionaltype((d: any) => d.checkType(T_).extendsType(
    (i: any) => i.tsinfertype((f: any) => f.typeParameter((p: any) => p.name((n: any) => n.name("W"))))).trueType(T_).falseType(U_)))));
assert(equal(scopesOf("tsconditionaltype", conditional), [[], [], ["V"], []]));
const nested = typed((t: any) => t.tsconditionaltype((c: any) => c.checkType(T_).extendsType((e: any) => e.tsconditionaltype((d: any) => d.checkType(T_)
  .extendsType((i: any) => i.tsinfertype((f: any) => f.typeParameter((p: any) => p.name((n: any) => n.name("W"))))).trueType(T_).falseType(U_)))
  .trueType(T_).falseType(U_)));
assert(scopesOf("tsconditionaltype", nested)[2]!.length === 0); // the inner one's own
for (const extend of [(i: any) => i.tsinfertype((f: any) => f.typeParameter((p: any) => p)), (i: any) => i.tsinfertype((f: any) => f), null]) {
  const written = (c: any) => extend !== null ? c.checkType(T_).extendsType(extend) : c.checkType(T_);
  const anonymous = typed((t: any) => t.tsconditionaltype((c: any) => written(c).trueType(T_).falseType(U_)));
  const found = scopesOf("tsconditionaltype", anonymous);
  assert(found[found.length - 2]!.length === 0); // an infer of no name, or none, binds none
}
// A mapped type with no key, or a key of no name, binds none; a type parameter of no name declares none
for (const key of [(m: any) => m, (m: any) => m.key((i: any) => i)]) {
  const unnamed = typed((t: any) => t.tsmappedtype((m: any) => key(m).value(K_).body(K_)));
  assert(equal(scopesOf("tsmappedtype", unnamed), [[], [], []]));
}
const anonymousParameters = typed((t: any) => t.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p, (p: any) => p.name((i: any) => i)]))));
assert(scopesOf("tsfunctiontype", anonymousParameters).every((s) => s.length === 0));
// A signature binds its parameters within it, through the type literal that holds it; a key, not computed, is a name
const literal = typed((t: any) => t.tstypeliteral((q: any) => q.members([
  (m: any) => m.tsmethodsignature((s: any) => s.key((k: any) => k.identifier((i: any) => i.name("m"))).params([x]).returnType(
    (r: any) => r.typeAnnotation((p: any) => p.tstypequery((e: any) => e.exprName((n: any) => n.identifier((i: any) => i.name("x"))))))),
  (m: any) => m.tsindexsignature((s: any) => s.parameters([(p: any) => p.identifier((i: any) => i.name("k"))])),
  (m: any) => m.tspropertysignature((s: any) => s.computed(true).key(x)),
  (m: any) => m.tscallsignaturedeclaration((s: any) => s.params([(p: any) => p.identifier((i: any) => i.name("y"))]))])));
assert(equal(scopesOf("tstypeliteral", literal), [["m", "x"], ["x"], ["x"], ["k"], [], ["y"]]));
assert(same(TST.DIALECT.validate(literal), ["members 2 key: identifier 'x' is not bound"])); // a computed key, a value's name
assert(same(names(SY.free(literal, TST.DIALECT)), ["x"]));

<a id="TST-04"></a>
## TST-04 · Its editions: the versions of TypeScript 5, 5.0 to 5.9, by their published names, each with every one of its own kinds

In [ ]:
const tst = store.Instances.Expressions.TypeScript.Types;
const VERSIONS = Array.from({ length: 10 }, (_, minor) => `TypeScript 5.${minor}`);
assert(same([...FS.standards(tst).keys()], VERSIONS) && TST.SINCE.size === 0);
assert(equal(FS.properties(FS.find(tst, "TypeScript 5.4")!), map({ bodies: ["com.microsoft"], revision: ["version", "5.4"] })));
assert(same(FS.kinds(FS.find(tst, "TypeScript 5.0")!).map((s: any) => s.name), TST.TAGS.map((tag) => kinds.get(tag).name))); // its own
store.transaction(TypeScriptTypes.define); // again: it writes nothing
assert(same([...FS.standards(tst).keys()], VERSIONS));

<a id="TST-05"></a>
## TST-05 · Printed as TypeScript writes types, by precedence, from its own holder and within TypeScript5's, by the family's printer

In [ ]:
assert(TSTText.ToText === TSText.ToText && same([...TSTText.TEXTS.keys()].sort(), [...TST.TAGS, "tspropertysignature",
  "tsmethodsignature", "tscallsignaturedeclaration", "tsconstructsignaturedeclaration", "tsindexsignature"].sort()));
assert(TSTText.PRECEDENCE.get("tsuniontype") === TSTText.UNION && TSTText.FUNCTION < TSTText.CONDITIONAL && TSTText.CONDITIONAL < TSTText.PRIMARY);
for (const [spec, text] of [
  [(t: any) => t.tsuniontype((u: any) => u.types([T_, (e: any) => e.tsfunctiontype((f: any) => f.returnType((r: any) => r.typeAnnotation(U_)))])),
    "T | (() => U)"], // a function type as an operand, parenthesized
  [(t: any) => t.tsmappedtype((m: any) => m.readonly("-readonly").key((i: any) => i.name("K")).value((e: any) => e.tstypeoperator(
    (o: any) => o.operator("keyof").typeAnnotation(T_))).optional("-?").body((e: any) => e.tsindexedaccesstype(
    (a: any) => a.objectType(T_).indexType(K_)))), "{ -readonly [K in keyof T]-?: T[K] }"],
  [(t: any) => t.tsconstructortype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).params([
    (p: any) => p.identifier((i: any) => i.name("a"))]).returnType((r: any) => r.typeAnnotation(T_))), "new <T>(a) => T"],
  [(t: any) => t.tstypeliteral((q: any) => q.members([
    (m: any) => m.tsmethodsignature((s: any) => s.key((k: any) => k.identifier((i: any) => i.name("m"))).params([x]).returnType(
      (r: any) => r.typeAnnotation(T_))),
    (m: any) => m.tsindexsignature((s: any) => s.readonly(true).parameters([(p: any) => p.identifier((i: any) => i.name("k").typeAnnotation(
      (a: any) => a.typeAnnotation((k: any) => k.tsstringkeyword((h: any) => h))))]).typeAnnotation((a: any) => a.typeAnnotation(T_)))])),
    "{ m(x): T; readonly [k: string]: T; }"],
  [(t: any) => t.tsconditionaltype((c: any) => c.checkType(T_).extendsType((e: any) => e.tsarraytype((a: any) => a.elementType(
    (i: any) => i.tsinfertype((f: any) => f.typeParameter((p: any) => p.name((n: any) => n.name("V"))))))).trueType(reference("V")).falseType(
    (e: any) => e.tsneverkeyword((h: any) => h))), "T extends (infer V)[] ? V : never"],
  [(t: any) => t.tsimporttype((i: any) => i.source((s: any) => s.literal((li: any) => li.value((v: any) => v.as_scalar("m")))).qualifier(
    (q: any) => q.tsqualifiedname((n: any) => n.left((l: any) => l.identifier((j: any) => j.name("A"))).right((j: any) => j.name("B"))))),
    'import("m").A.B'],
  [(t: any) => t.error((e: any) => e.text("T T")), "T T"], // invalid syntax, as written
] as [unknown, string][]) {
  assert(TSText.ToText(typed(spec)) === text, globalThis.JSON.stringify([TSText.ToText(typed(spec)), text]));
  assert(TSText.ToText(held(spec)) === text); // from its holder
}
raises(ValueError, () => TSText.ToText(typed((t: any) => t.error((e: any) => e))), "TypeScript Types error has no text to print"); // by
// its dialect's name
// Within TypeScript5's terms, an annotation, a type argument, and a type parameter's constraint and default
assert(TSText.ToText(ts((u: any) => u.arrowfunctionexpression((a: any) => a.typeParameters((d: any) => d.params([
  (p: any) => p.name((i: any) => i.name("T")).constraint((e: any) => e.tsuniontype((n: any) => n.types([number, U_]))).default(number)]))
  .params([(p: any) => p.identifier((i: any) => i.name("v").typeAnnotation((b: any) => b.typeAnnotation(T_)))]).returnType(
    (r: any) => r.typeAnnotation((e: any) => e.tsfunctiontype((f: any) => f.returnType((s: any) => s.typeAnnotation(T_)))))
  .body((b: any) => b.callexpression((c: any) => c.callee(x).typeArguments((g: any) => g.params([T_]))))))) ===
  "<T extends number | U = number>(v: T): () => T => x<T>()");